# [5.2] Gemma Scope and Feature Steering - Exercises

This section turns sparse-feature interpretability into a falsifiable experimental pipeline. You will write the small pieces that make a Gemma Scope-style result inspectable: sparsity metrics, reconstruction metrics, direct logit attribution, held-out feature validation, ablation, and steering controls.

The real-model path is intentionally separated from the implementation exercises. The exercises run quickly on CPU-sized tensors; the committed verification report then checks the pinned Gemma Scope 2 1B-IT layer-13 SAE artifact and authenticated Gemma 3 activations on CUDA.

<img src="../../instructions/assets/gemma_scope_validation_ladder.svg" width="820">

By the end, you should be able to distinguish three claims:

1. a sparse feature fires often enough to inspect,
2. the feature predicts a held-out semantic split better than matched controls,
3. an intervention through the decoder vector changes the target behavior more than a random-direction control.


In [ ]:
import json
import sys
from dataclasses import dataclass
from pathlib import Path
from typing import Literal

import torch as t
import torch.nn.functional as F

chapter = "chapter5_modern_architectures"
section = "part2_gemma_scope_feature_steering"
root_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / chapter).exists())
if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))

exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

import part2_gemma_scope_feature_steering.tests as tests
import part2_gemma_scope_feature_steering.utils as utils
from arena_ext.gemma_scope import gemma_scope_artifact_preflight

MAIN = __name__ == "__main__"


@dataclass(frozen=True)
class SAEReconstructionMetrics:
    l0: float
    feature_density_mean: float
    dead_feature_fraction: float
    reconstruction_mse: float
    reconstruction_kl: float | None = None
    loss_recovered: float | None = None


@dataclass(frozen=True)
class FeatureDetectionReport:
    auc: float
    positive_mean: float
    negative_mean: float
    separation: float
    threshold_accuracy: float


@dataclass(frozen=True)
class SteeringComparisonReport:
    baseline_mean: float
    steered_mean: float
    random_mean: float
    steered_delta: float
    random_delta: float
    passes_control: bool


## 1. Sparsity Metrics

A sparse autoencoder is not useful just because it reconstructs activations. It should also use a small, inspectable set of features. Your first task is to compute firing-rate statistics over all non-feature axes.

Implement:

- `feature_density`: firing rate per feature,
- `l0`: average number of active features per activation vector,
- `dead_feature_fraction`: fraction of features that never fire.

<details><summary>Expected output</summary>

The test should print:

```text
All tests in `test_feature_density_l0_and_dead_fraction_match_reference` passed!
```

For the test tensor, feature 1 is dead, features 0 and 2 fire on half the positions, and the average active-feature count is 1.25.

</details>

<details><summary>Help - reducing over the right axes</summary>

The last axis indexes features. Every earlier axis is an example-like axis: batch, sequence position, or any other grouping. Convert activations to a boolean firing mask, then average over every axis except the final one.

</details>


In [ ]:
def feature_density(feature_acts: t.Tensor, threshold: float = 0.0) -> t.Tensor:
    """Return the firing rate for each feature over all non-feature dimensions."""
    raise NotImplementedError()


def l0(feature_acts: t.Tensor, threshold: float = 0.0) -> float:
    """Return the average number of active features per activation vector."""
    raise NotImplementedError()


def dead_feature_fraction(feature_acts: t.Tensor, threshold: float = 0.0) -> float:
    """Return the fraction of features that never fire."""
    raise NotImplementedError()


tests.test_feature_density_l0_and_dead_fraction_match_reference(
    feature_density,
    l0,
    dead_feature_fraction,
)


## 2. Reconstruction Metrics

SAE papers usually report several reconstruction measures together. You will combine activation MSE, logit KL, loss recovered, and the sparsity metrics from the previous exercise into one dataclass.

The important habit is to keep reconstruction and sparsity in the same table. A dense autoencoder can reconstruct well while explaining very little.

<details><summary>Expected output</summary>

The test should print:

```text
All tests in `test_compute_sae_reconstruction_metrics_matches_manual_and_reference` passed!
```

The manual reconstruction MSE in the test is computed directly from `(reconstructed - activations) ** 2`, and `loss_recovered` should equal `0.75`.

</details>

<details><summary>Help - interpreting loss recovered</summary>

Use

```python
(zero_ablation_loss - reconstructed_loss) / (zero_ablation_loss - clean_loss)
```

A value near 1 means the reconstruction recovers most of the clean model's performance relative to zero ablation. A value near 0 means it behaves more like removing the activation entirely.

</details>


In [ ]:
def mean_kl_divergence(reference_logits: t.Tensor, reconstructed_logits: t.Tensor) -> float:
    """Compute mean KL(reference || reconstructed) across all non-vocab axes."""
    raise NotImplementedError()


def loss_recovered(
    *,
    clean_loss: float,
    reconstructed_loss: float,
    zero_ablation_loss: float,
) -> float:
    """Return the usual SAE loss-recovered score."""
    raise NotImplementedError()


def compute_sae_reconstruction_metrics(
    *,
    activations: t.Tensor,
    reconstructed_activations: t.Tensor,
    feature_acts: t.Tensor,
    threshold: float = 0.0,
    reference_logits: t.Tensor | None = None,
    reconstructed_logits: t.Tensor | None = None,
    clean_loss: float | None = None,
    reconstructed_loss: float | None = None,
    zero_ablation_loss: float | None = None,
) -> SAEReconstructionMetrics:
    """Compute reconstruction, sparsity, KL, and loss-recovered metrics."""
    raise NotImplementedError()


tests.test_compute_sae_reconstruction_metrics_matches_manual_and_reference(
    compute_sae_reconstruction_metrics,
)


## 3. Direct Logit Attribution

Direct logit attribution asks: if a sparse feature contributes decoder vector `d` to the residual stream, which vocabulary logits does that vector directly move through the unembedding?

This is a useful first-pass inspection tool. It is not causal evidence by itself.

<details><summary>Expected output</summary>

The test should print:

```text
All tests in `test_direct_logit_attribution_projects_decoder_vectors` passed!
```

For the fixture, selecting tokens `[0, 2]` should return

```python
[[1.0, 3.0], [4.0, 6.0], [-2.0, 0.0]]
```

</details>

<details><summary>Help - why DLA is not enough</summary>

DLA ignores the rest of the network. A decoder vector can point at a token through the unembedding while the model's downstream computation suppresses, routes around, or reverses the effect. Treat DLA as a hypothesis generator, then test the hypothesis with held-out examples and interventions.

</details>


In [ ]:
def direct_logit_attribution(
    decoder_vectors: t.Tensor,
    unembedding: t.Tensor,
    token_ids: t.Tensor | list[int] | None = None,
) -> t.Tensor:
    """Project feature decoder vectors through the unembedding matrix."""
    raise NotImplementedError()


tests.test_direct_logit_attribution_projects_decoder_vectors(direct_logit_attribution)


## 4. Held-Out Feature Validation

Top-activating examples are not a validation set. In this exercise, you score a feature on positives and matched negatives that were not used to choose the feature.

Implement a tie-aware binary ROC AUC and a compact report with positive mean, negative mean, separation, and threshold accuracy.

<details><summary>Expected output</summary>

The test should print:

```text
All tests in `test_feature_detection_report_handles_heldout_controls` passed!
```

The perfectly separated fixture should have AUC `1.0` and threshold accuracy `1.0`; the tied-score fixture checks that average ranks are handled correctly.

</details>

<details><summary>Help - average ranks for ties</summary>

Sort scores in ascending order, assign 1-indexed ranks, and give tied values the average rank they span. The AUC is the probability that a random positive receives a higher score than a random negative, with ties counted halfway.

</details>


In [ ]:
def roc_auc_binary(scores: t.Tensor, labels: t.Tensor) -> float:
    """Compute binary ROC AUC using average ranks, including ties."""
    raise NotImplementedError()


def feature_detection_report(
    scores: t.Tensor,
    labels: t.Tensor,
    threshold: float | None = None,
) -> FeatureDetectionReport:
    """Evaluate a feature against held-out positives and matched negatives."""
    raise NotImplementedError()


tests.test_feature_detection_report_handles_heldout_controls(
    feature_detection_report,
    roc_auc_binary,
)


## 5. Top Activations and Ablation

Now implement two inspection utilities. `top_activating_examples` tells you which flattened positions fired most strongly for a feature. `ablate_features` constructs a controlled feature-activation tensor where selected features are replaced by zero or by their global mean.

<details><summary>Expected output</summary>

The test should print:

```text
All tests in `test_top_activating_examples_and_ablation_match_reference` passed!
```

Feature 3 in the synthetic batch is planted to fire strongly in the first half of examples, so top-k should return positions from that region, and zero ablation should set every selected activation to zero.

</details>

<details><summary>Help - flattened positions are deliberate</summary>

Return flattened indices rather than `(batch, position)` pairs. That keeps the utility independent of whether the caller is using text positions, image patches, or some other axis layout. The caller can use `torch.unravel_index` later if it wants structured coordinates.

</details>


In [ ]:
def top_activating_examples(
    feature_acts: t.Tensor,
    feature_id: int,
    k: int = 10,
) -> tuple[t.Tensor, t.Tensor]:
    """Return flattened positions and values for the top activations of one feature."""
    raise NotImplementedError()


def ablate_features(
    feature_acts: t.Tensor,
    feature_ids: t.Tensor | list[int],
    replacement: Literal["zero", "mean"] = "zero",
) -> t.Tensor:
    """Ablate selected sparse feature activations."""
    raise NotImplementedError()


tests.test_top_activating_examples_and_ablation_match_reference(
    top_activating_examples,
    ablate_features,
)


## 6. Decoder Steering Controls

Feature steering adds one or more SAE decoder directions to the residual stream. The effect is only meaningful if it beats a matched random-direction or random-feature control.

Implement steering at either the last position or every position, then summarize the target-score shift against a random control.

<details><summary>Expected output</summary>

The test should print:

```text
All tests in `test_decoder_steering_and_random_control_report` passed!
```

The fixture checks both last-position and all-position steering, scalar and vector coefficients, and a control report where the feature direction beats the random-control delta.

</details>

<details><summary>Help - what counts as a meaningful steering result?</summary>

A good steering result changes the intended score more than a random direction with the same norm and placement. It should also leave unrelated behavior reasonably stable. This notebook only checks the first requirement; larger labs should add capability and side-effect measurements.

</details>


In [ ]:
def apply_decoder_steering(
    activations: t.Tensor,
    decoder_vectors: t.Tensor,
    feature_ids: t.Tensor | list[int],
    coefficients: t.Tensor | list[float] | float,
    *,
    positions: Literal["all", "last"] = "last",
) -> t.Tensor:
    """Add selected feature decoder directions to the residual stream."""
    raise NotImplementedError()


def steering_comparison_report(
    baseline_scores: t.Tensor,
    steered_scores: t.Tensor,
    random_control_scores: t.Tensor,
) -> SteeringComparisonReport:
    """Compare feature steering against a random-feature control."""
    raise NotImplementedError()


tests.test_decoder_steering_and_random_control_report(
    apply_decoder_steering,
    steering_comparison_report,
)


## 7. Whole-Notebook Contract

Run this after you have passed the individual exercises. It checks the expected public contract of the section's implementation, not just one helper at a time.

<details><summary>Expected output</summary>

The test should print:

```text
All tests in `test_notebook_contract` passed!
```

This is still a small deterministic CPU smoke test. The real CUDA evidence is in `verification_report.json` and is summarized below.

</details>

<details><summary>Help - why keep this small?</summary>

The notebook tests should be fast enough to run while learning. The expensive model and artifact checks belong in the committed verification report, where their environment, input hashes, and GPU metrics are recorded.

</details>


In [ ]:
# Run after the previous cells pass.
tests.test_notebook_contract()


## Signature Result

The committed report for this section records the local CUDA verification target. You should read this table as the section's evidence boundary: it proves the feature-validation ladder and artifact path work locally, while avoiding a broad claim that all Gemma Scope features are understood.

| Check | Required result |
|---|---:|
| Gemma Scope artifact preflight | `true` |
| Gemma Scope forward pass | `true` |
| SAE width | `16384` |
| Residual width | `1152` |
| Held-out feature AUC | `1.000` |
| Random-feature baseline AUC | `0.500` |
| Label-shuffle control | `true` |
| Peak VRAM | `< 24 GB` |

<details><summary>Interpreting the signature result</summary>

The report says the pinned released SAE artifact loads, has the expected tensor shapes, runs encode/decode on CUDA, and that the real activation validation beats random-feature and label-shuffle controls. It does not say that a single top activation is an explanation, and it does not replace a broader feature taxonomy audit.

</details>

<details><summary>Help - reading the JSON report</summary>

The `metrics.gpu_test` object contains the CUDA evidence. The `accepted` and `tests` flags are the release gate; the `input_hashes` object records the files whose contents were verified when the report was written.

</details>


In [ ]:
def _load_committed_gpu_report() -> dict:
    report_path = section_dir / "verification_report.json"
    report = json.loads(report_path.read_text())
    gpu = report["metrics"]["gpu_test"]
    return {
        "accepted": report["accepted"],
        "tests": report["tests_passed"],
        "device": gpu["device"],
        "torch": gpu["torch_version"],
        "cuda": gpu["cuda_version"],
        "artifact_preflight": gpu["gemma_scope_artifact_preflight_passed"],
        "forward_pass": gpu["gemma_scope_forward_passed"],
        "width": gpu["gemma_scope_width"],
        "d_model": gpu["gemma_scope_d_model"],
        "feature_auc": gpu["gemma_scope_real_activation_feature_auc"],
        "baseline_auc": gpu["gemma_scope_real_activation_baseline_auc"],
        "label_shuffle_control": gpu["gemma_scope_real_activation_label_shuffle_control_passed"],
        "peak_vram_gb": gpu["peak_vram_gb"],
    }

def run_gpu_test(max_vram_gb: float = 24.0) -> dict:
    gpu = _load_committed_gpu_report()
    assert gpu["accepted"] and gpu["tests"]
    assert gpu["peak_vram_gb"] <= max_vram_gb
    return gpu


def run_full_experiment(max_vram_gb: float = 24.0) -> dict:
    return run_gpu_test(max_vram_gb=max_vram_gb)


run_gpu_test()


## Limitations

This section teaches the validation ladder for feature-level interpretability. It does not claim a complete audit of Gemma Scope, a catalog of all layer-13 features, or a safety evaluation of Gemma 3. The intended standard is narrower and more testable: when you name a feature, you must show held-out separation and intervention/control evidence before treating it as meaningful.
